# 11 · Spark + HDFS — Datalake em HDFS via Spark Connect

**Teoria**: docs/07-spark-e-hdfs.md

**Pré-requisito**: `make spark && make hadoop` em execução.

🎯 **Objetivo**: demonstrar o Spark cluster consumindo o datalake via **HDFS nativo** (`hdfs://namenode:8020`). Diferente do S3 (object store), o HDFS é um sistema de arquivos distribuído com localidade de dados, replicação e operações de rename atômicas.

---
### 🔤 O que você vai praticar

1. Upload de dados locais para o HDFS via gateway HttpFS (`http://localhost:14000`)
2. Leitura de Parquet via `hdfs://` — `spark.read.parquet("hdfs://namenode:8020/datalake/bronze/vendas")`
3. Select / Filter / WithColumn → **Silver** layer, escrita no HDFS
4. GroupBy + Agg + OrderBy — agregações de negócio direto no HDFS
5. Broadcast Join + Window Functions → **Gold** layer
6. Spark SQL com `LOCATION 'hdfs://...'`
7. CSV e JSON também no HDFS — formatos não-Parquet
8. **Bônus**: provar replicação HDFS

Vamos começar populando o HDFS com os dados bronze.

In [ ]:
from pathlib import Path

import requests

bronze_dir = Path("../data/bronze")

BASE = "http://localhost:14000/webhdfs/v1"
HDFS_ROOT = "/datalake/bronze"

session = requests.Session()

def upload_to_hdfs(local_path: Path) -> str:
    rel = local_path.relative_to(bronze_dir)
    hdfs_path = f"{HDFS_ROOT}/{rel.as_posix()}"
    dir_part = "/".join(hdfs_path.split("/")[:-1])
    session.put(f"{BASE}{dir_part}?op=MKDIRS&user.name=root")
    # CREATE do WebHDFS é em 2 passos: o 1º PUT (sem corpo) responde 307 com o
    # Location real para onde os bytes devem ir. Não podemos deixar o requests
    # seguir o redirect automaticamente, pois o arquivo já teria sido consumido
    # (EOF) e o 2º PUT seguiria sem corpo — e o HttpFS exige Content-Type
    # application/octet-stream nesse 2º PUT, senão responde 400 Bad Request.
    redirect = session.put(f"{BASE}{hdfs_path}?op=CREATE&user.name=root&overwrite=true", allow_redirects=False)
    redirect.raise_for_status()
    with open(local_path, "rb") as f:
        resp = session.put(
            redirect.headers["Location"],
            data=f.read(),
            headers={"Content-Type": "application/octet-stream"},
        )
    resp.raise_for_status()
    return hdfs_path

files = sorted([f for f in bronze_dir.rglob("*") if f.is_file()])
n = len(files)
print(f"📤 Enviando {n} arquivo(s) para hdfs:///datalake/bronze/ ...\n")

for i, fp in enumerate(files, 1):
    path = upload_to_hdfs(fp)
    print(f"📤 Upload [{i:2d}/{n}] hdfs://{path}")

print(f"\n✅ Upload concluído. {n} arquivo(s) em hdfs:///datalake/bronze/")

# Lista o diretório bronze
resp = requests.get(f"{BASE}/datalake/bronze?op=LISTSTATUS&user.name=root")
if resp.ok:
    print("\n📦 Pastas em /datalake/bronze:")
    for entry in resp.json()["FileStatuses"]["FileStatus"]:
        print(f"   📁  {entry['pathSuffix']}/")

---
### 🧠 O conceito: Spark + HDFS nativo

Nos notebooks 06-09, o pipeline usava volume Docker ou S3. **Aqui é diferente.**

Neste perfil **hadoop**:
- O HDFS roda em containers separados (namenode + 2 datanodes)
- O Spark Connect HDFS roda em container separado (spark-connect-hdfs), reutilizando o mesmo master/workers do profile cluster
- A comunicação Spark ↔ HDFS usa o **protocolo nativo HDFS RPC** (`hdfs://namenode:8020`), sem HttpFS intermediário
- O upload inicial é que passa pelo **gateway HttpFS** (porta 14000), porque o host não enxerga a rede interna do HDFS
- O Spark Connect usa `sc://localhost:15004` — porta diferente (15004) do perfil cluster, com Spark Connect configurado com `spark.hadoop.fs.defaultFS=hdfs://namenode:8020`

💡 Essa é a arquitetua real de ambientes on-premises que separam compute (Spark) de storage (HDFS).

Vamos conectar.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15004")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  NameNode UI:  http://localhost:9870")
print("📊 Spark App UI: http://localhost:4042")
print("🔌 Spark Connect: sc://localhost:15004  (HDFS nativo)")

---
### 📌 Lendo o Bronze do HDFS

A leitura usa o mesmo `spark.read.parquet()`, mas o caminho agora é `hdfs://` — o Hadoop HDFS connector resolve a comunicação com o NameNode via RPC.

💡 O segredo está nas `--conf spark.hadoop.fs.defaultFS=hdfs://namenode:8020` que o `spark-connect-hdfs` carrega no bootstrap.

In [ ]:
sdf_vendas = spark.read.parquet("hdfs://namenode:8020/datalake/bronze/vendas")
sdf_funcionarios = spark.read.parquet("hdfs://namenode:8020/datalake/bronze/funcionarios")
sdf_empresas = spark.read.parquet("hdfs://namenode:8020/datalake/bronze/empresas")

print("📦 vendas")
sdf_vendas.printSchema()
print(f"   Registros: {sdf_vendas.count():,}")
sdf_vendas.show(5)

print("📦 empresas")
sdf_empresas.show(5)

print("📦 funcionarios")
sdf_funcionarios.show(5)

---
### Select / Filter / WithColumn → Camada Silver

Mesmas transformações dos notebooks anteriores, agora escrevendo no HDFS.

Vamos enriquecer as vendas com uma faixa de valor (segmentação) e filtrar registros inconsistentes, gravando o resultado particionado por ano.

In [ ]:
from pyspark.sql.functions import col, when
from pyspark.sql.functions import sum as spark_sum

vendas_silver = (
    sdf_vendas.filter(col("valor") > 0)
    .withColumn("faixa_valor",
        when(col("valor") < 200, "baixo")
        .when(col("valor") < 1000, "medio")
        .otherwise("alto"))
)

(vendas_silver
    .write.mode("overwrite").partitionBy("ano")
    .parquet("hdfs://namenode:8020/datalake/silver/vendas_enriquecidas"))

sdf_silver = spark.read.parquet("hdfs://namenode:8020/datalake/silver/vendas_enriquecidas")
print(f"✅ Silver escrita: {sdf_silver.count():,} registros em hdfs://namenode:8020/datalake/silver/vendas_enriquecidas")
sdf_silver.select("id_venda", "id_funcionario", "valor", "faixa_valor", "ano", "mes").show(10)

---
### GroupBy + Agg — Análises de Negócio

Agregações clássicas: total de vendas por mês, ticket médio, volume de transações.

📌 Tudo lido e processado diretamente do HDFS — o Spark baixa só os arquivos necessários (predicate pushdown via partição `ano`/`mes`).

In [ ]:
from pyspark.sql.functions import avg, count
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import round as spark_round

resumo_mensal = (
    sdf_vendas.groupBy("ano", "mes")
    .agg(
        spark_sum("valor").alias("total_vendas"),
        count("*").alias("numero_vendas"),
        spark_round(avg("valor"), 2).alias("ticket_medio"),
        spark_max("valor").alias("maior_venda"),
    )
    .orderBy("ano", "mes")
)

print("📊 Resumo mensal de vendas:")
resumo_mensal.show(15)

# Escreve como camada exploratória no HDFS
(resumo_mensal
    .write.mode("overwrite").partitionBy("ano")
    .parquet("hdfs://namenode:8020/datalake/silver/resumo_mensal"))

print("✅ Resumo mensal salvo em hdfs://namenode:8020/datalake/silver/resumo_mensal")

---
### Broadcast Join + Window Functions → Camada Gold

Pipeline completo: juntar vendas com empresas (broadcast, já que `empresas` cabe na memória), rankear setores por período, e gravar a **Gold layer** — pronta para dashboards e consultas analíticas.

🧠 Broadcast join evita shuffle: o Spark copia `sdf_empresas` para cada executor e faz o merge localmente.

In [ ]:
from pyspark.sql.functions import broadcast, row_number
from pyspark.sql.window import Window

vendas_com_setor = (
    sdf_vendas
    .join(broadcast(sdf_empresas), "id_empresa")
)

# Agregação por setor + período
gold_agregado = (
    vendas_com_setor.groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)

print("🏆 Top 10 setores por volume de vendas:")
gold_agregado.show(10)

# Ranking por período (Window)
janela_top = Window.partitionBy("ano", "mes").orderBy(col("total_vendas").desc())
top_setores = (
    gold_agregado
    .withColumn("posicao", row_number().over(janela_top))
    .filter(col("posicao") <= 3)
)

(top_setores
    .write.mode("overwrite").partitionBy("ano", "mes")
    .parquet("hdfs://namenode:8020/datalake/gold/top_setores"))

print("✅ Gold layer escrita em hdfs://namenode:8020/datalake/gold/top_setores")

---
### 🎯 Spark SQL sobre dados do HDFS

O Spark SQL funciona perfeitamente com HDFS. A sintaxe `LOCATION 'hdfs://...'` cria uma tabela externa apontando direto para o sistema de arquivos distribuído.

💡 A diferença para o S3: no HDFS, as operações de rename são atômicas (o que torna `OVERWRITE` confiável), e o Spark pode escalonar executors com localidade de dados (data locality).

In [ ]:
# Registra os DataFrames como views temporárias
sdf_vendas.createOrReplaceTempView("vendas")
sdf_empresas.createOrReplaceTempView("empresas")

# Cria uma tabela externa apontando para a Silver no HDFS
spark.sql("""
    CREATE OR REPLACE TEMP VIEW silver_vendas
    USING parquet
    OPTIONS (path 'hdfs://namenode:8020/datalake/silver/vendas_enriquecidas')
""").show()

# Consulta SQL sobre dados no HDFS
spark.sql("""
    SELECT e.setor, v.ano, v.mes,
           ROUND(SUM(v.valor), 2) AS total
    FROM vendas v
    JOIN empresas e ON v.id_empresa = e.id_empresa
    GROUP BY e.setor, v.ano, v.mes
    ORDER BY total DESC
    LIMIT 10
""").show()

# Explica o plano Catalyst — veja o predicate pushdown!
print("📋 Plano Catalyst da consulta:")
spark.sql("""
    SELECT faixa_valor, ROUND(SUM(valor), 2) AS total
    FROM silver_vendas
    WHERE ano = 2025
    GROUP BY faixa_valor
    ORDER BY total DESC
""").explain(True)

---
### 🎁 Bônus: CSV e JSON também no HDFS

O HDFS não serve só para Parquet. Vamos ler as avaliações diretamente — exatamente como nos notebooks anteriores, mas agora via `hdfs://`.

📌 Os CSVs com `sep=";"`, `encoding="ISO-8859-1"` e data em formato brasileiro funcionam normalmente.

In [ ]:
from pyspark.sql.functions import col

# JSON Lines (app) — campo aninhado
sdf_app = spark.read.json("hdfs://namenode:8020/datalake/bronze/avaliacoes_app")

sdf_app.select("id_avaliacao", "nota", col("dispositivo.os").alias("os")).show(5)

# CSV limpo (site)
sdf_site = (spark.read.option("header", True).option("inferSchema", True)
    .csv("hdfs://namenode:8020/datalake/bronze/avaliacoes_site"))
print("\n📄 Site: {} avaliações".format(sdf_site.count()))
sdf_site.show(5)

# CSV legado (call center) — sep, encoding, data br
sdf_cc = (spark.read.option("header", True)
    .option("sep", ";").option("encoding", "ISO-8859-1").option("inferSchema", True)
    .csv("hdfs://namenode:8020/datalake/bronze/avaliacoes_callcenter"))
print("\n📞 Call Center: {} avaliações".format(sdf_cc.count()))
sdf_cc.show(5)

# Unifica os 3 canais (unionByName)
sdf_app_norm = sdf_app.select("id_avaliacao", "id_empresa", "nota")
sdf_site_norm = sdf_site.select("id_avaliacao", "id_empresa", "nota")
sdf_cc_norm = sdf_cc.select("id_avaliacao", "id_empresa", "nota")

sdf_todas = (sdf_app_norm
    .unionByName(sdf_site_norm, allowMissingColumns=True)
    .unionByName(sdf_cc_norm, allowMissingColumns=True))

print("\n📊 Total de avaliações unificadas: {}".format(sdf_todas.count()))


---
### 💪 A prova da replicação HDFS

Diferente do S3 (que replica por erasure coding), o HDFS replica blocos inteiros entre os datanodes.

Podemos verificar o fator de replicação dos arquivos que acabamos de escrever:

```bash
# No shell do namenode:
docker compose --profile hadoop exec namenode hdfs dfs -ls -R /datalake
docker compose --profile hadoop exec namenode hdfs dfs -stat "%r" /datalake/gold/top_setores/ano=2026/mes=07/*.parquet
```

O fator de replicação padrão é 3 (mas com 2 datanodes, o HDFS replica em todos os disponíveis).

In [ ]:
print("📖 Lendo Gold layer do HDFS...")
sdf_gold = spark.read.parquet("hdfs://namenode:8020/datalake/gold/top_setores")
print(f"   Registros: {sdf_gold.count():,}")
sdf_gold.orderBy("ano", "mes", "posicao").show(15)

# Mostra que o Spark empurra filtros para o HDFS (predicate pushdown)
print("📋 Plano Catalyst — veja o pushdown dos filtros:")
sdf_gold.filter(col("ano") == 2025).select("setor", "total_vendas").explain(True)

---
🎉 **Parabéns!** Você completou o notebook 11.

Você aprendeu:
- Fazer upload de dados locais para o HDFS via gateway HttpFS (`requests.put`)
- Ler e escrever Parquet no HDFS com `hdfs://namenode:8020/datalake/...`
- Executar **select / filter / withColumn / groupBy / agg / orderBy / join / window** — tudo lendo e escrevendo em HDFS
- Construir as 3 camadas **Bronze → Silver → Gold** no HDFS via Spark Connect
- Usar **Spark SQL** com `LOCATION 'hdfs://...'`

🧠 **Moral da história:** o Spark abstrai completamente o sistema de armazenamento. O mesmo pipeline Bronze→Silver→Gold roda igual em disco local, volume Docker, S3 (`s3a://`) e HDFS (`hdfs://`). O que muda é o prefixo do caminho.

📌 **Comparação rápida:**
- `s3a://` → object store (RustFS). Escalável, barato, sem localidade. Ideal para cloud.
- `hdfs://` → filesystem distribuído. Localidade de dados, rename atômico. Ideal para on-premises.
- Todos usam **Spark Connect** — o que muda é a porta (15002 volume, 15003 S3, 15004 HDFS).

---
**Próximo passo**: feche a sessão com `spark.stop()` e siga para o notebook 13 (Grand Benchmark), que executa o pipeline Gold nas 4 arquiteturas lado a lado.
